In [1]:
from __future__ import print_function

import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import SGD, Adam, Adadelta
from tensorflow.keras.activations import relu
from tensorflow.keras.regularizers import l2
from tensorflow.keras.constraints import max_norm
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import mnist, cifar10
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ModelCheckpoint

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

print("Packages Loaded")

Packages Loaded


In [2]:
# Lets import our MNIST data like we did in lab 3.
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

In [3]:
(x_train_cifar, y_train_cifar), (x_test_cifar, y_test_cifar) = cifar10.load_data()
a = x_train_cifar.reshape(50000, -1)
a.shape

(50000, 3072)

In [4]:
# Now import Cifar-10 data and process it.
(x_train_cifar, y_train_cifar), (x_test_cifar, y_test_cifar) = cifar10.load_data()
x_train_cifar = x_train_cifar.reshape(50000, -1)
x_test_cifar = x_test_cifar.reshape(10000, -1)
x_train_cifar = x_train_cifar.astype('float32') / 255
x_test_cifar = x_test_cifar.astype('float32') / 255

y_test_cifar_origin = y_test_cifar
y_train_cifar = tf.keras.utils.to_categorical(y_train_cifar, 10)
y_test_cifar = tf.keras.utils.to_categorical(y_test_cifar, 10)
input_shape_cifar = x_train_cifar.shape[1:]

In [7]:
%%time
# Our baseline model for this lab.
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(784,)))
model.add(Dense(64, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(10, activation='softmax'))
model.compile(loss='categorical_crossentropy',
              optimizer=SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True),
              metrics=['accuracy'])
model.fit(x_train, y_train, epochs=20, batch_size=64, validation_data=(x_test, y_test))

Train on 60000 samples, validate on 10000 samples
Epoch 1/20
60000/60000 [==============================] - 4s 68us/step - loss: 0.4140 - acc: 0.8731 - val_loss: 0.1674 - val_acc: 0.9497
Epoch 2/20
60000/60000 [==============================] - 3s 49us/step - loss: 0.1521 - acc: 0.9544 - val_loss: 0.1238 - val_acc: 0.9626
Epoch 3/20
60000/60000 [==============================] - 3s 47us/step - loss: 0.1107 - acc: 0.9662 - val_loss: 0.1071 - val_acc: 0.9655
Epoch 4/20
60000/60000 [==============================] - 3s 45us/step - loss: 0.0876 - acc: 0.9724 - val_loss: 0.1027 - val_acc: 0.9680
Epoch 5/20
60000/60000 [==============================] - 3s 45us/step - loss: 0.0726 - acc: 0.9772 - val_loss: 0.0955 - val_acc: 0.9705
Epoch 6/20
60000/60000 [==============================] - 3s 47us/step - loss: 0.0612 - acc: 0.9807 - val_loss: 0.0902 - val_acc: 0.9731
Epoch 7/20
60000/60000 [==============================] - 3s 48us/step - loss: 0.0531 - acc: 0.9827 - val_loss: 0.0935 - val_acc

### Now convert this baseline into a Functional Model using keras' Functional Model API.
https://keras.io/getting-started/functional-api-guide/

In [8]:
%%time
# Create the functional Baseline here.
inputs = Input(shape=(784,))
output = Dense(64, activation='relu')(inputs)
output = Dense(64, activation='relu')(output)
output = Dense(64, activation='relu')(output)
output = Dense(64, activation='relu')(output)
predictions = Dense(10, activation='softmax')(output)
model = Model(inputs=inputs, outputs=predictions)
model.compile(optimizer=SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=20, batch_size=64, validation_data=(x_test, y_test))

Train on 60000 samples, validate on 10000 samples
Epoch 1/20
60000/60000 [==============================] - 3s 54us/step - loss: 0.3914 - acc: 0.8826 - val_loss: 0.1861 - val_acc: 0.9440
Epoch 2/20
60000/60000 [==============================] - 3s 46us/step - loss: 0.1473 - acc: 0.9559 - val_loss: 0.1385 - val_acc: 0.9579
Epoch 3/20
60000/60000 [==============================] - 3s 47us/step - loss: 0.1085 - acc: 0.9666 - val_loss: 0.1066 - val_acc: 0.9686
Epoch 4/20
60000/60000 [==============================] - 3s 46us/step - loss: 0.0867 - acc: 0.9730 - val_loss: 0.1007 - val_acc: 0.9696
Epoch 5/20
60000/60000 [==============================] - 3s 46us/step - loss: 0.0733 - acc: 0.9775 - val_loss: 0.1016 - val_acc: 0.9684
Epoch 6/20
60000/60000 [==============================] - 3s 47us/step - loss: 0.0604 - acc: 0.9808 - val_loss: 0.0999 - val_acc: 0.9713
Epoch 7/20
60000/60000 [==============================] - 3s 49us/step - loss: 0.0539 - acc: 0.9831 - val_loss: 0.1176 - val_acc

### Now we are going to compare our baseline to a shallow ResNet that we talked about in class. 
Pay attention to the changes we have made so far including optimizers, batch size, layer neuron width, and dropout.
Why did we make these changes? 

In [5]:
%%time
inputs = tf.keras.Input(shape=(784,), name='img')
x = Dense(128, activation='relu')(inputs)
block_1_output = Dense(128, activation='relu')(x)

x = Dense(128)(block_1_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
block_2_output = tf.keras.layers.add([x, block_1_output])

x = Dense(128)(block_2_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
block_3_output = tf.keras.layers.add([x, block_2_output])

x = Dense(128)(block_3_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
block_4_output = tf.keras.layers.add([x, block_3_output])

x = Dense(128, activation='relu')(block_4_output)
x = Dropout(0.5)(x)
outputs = Dense(10, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name='resnet')

model.compile(Adam(amsgrad=True), 'binary_crossentropy', metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=128,
          epochs=20,
          validation_data=(x_test, y_test))

Train on 60000 samples, validate on 10000 samples
Epoch 1/20
60000/60000 [==============================] - 11s 177us/step - loss: 0.1015 - acc: 0.9663 - val_loss: 0.0305 - val_acc: 0.9896
Epoch 2/20
60000/60000 [==============================] - 7s 118us/step - loss: 0.0363 - acc: 0.9888 - val_loss: 0.0247 - val_acc: 0.9918
Epoch 3/20
60000/60000 [==============================] - 7s 118us/step - loss: 0.0255 - acc: 0.9921 - val_loss: 0.0197 - val_acc: 0.9936
Epoch 4/20
60000/60000 [==============================] - 7s 118us/step - loss: 0.0208 - acc: 0.9935 - val_loss: 0.0158 - val_acc: 0.9951
Epoch 5/20
60000/60000 [==============================] - 7s 118us/step - loss: 0.0167 - acc: 0.9948 - val_loss: 0.0170 - val_acc: 0.9949
Epoch 6/20
60000/60000 [==============================] - 7s 117us/step - loss: 0.0135 - acc: 0.9957 - val_loss: 0.0158 - val_acc: 0.9953
Epoch 7/20
60000/60000 [==============================] - 8s 126us/step - loss: 0.0124 - acc: 0.9961 - val_loss: 0.0165 -

In [6]:
plot_model(model, to_file='reports/resnet.png')

### Now lets make a deeper ResNet. Make A network with 10 Residual Blocks. 
How does this affect training speed, accuracy, and stability?

In [25]:
def create_resnet(layers=3, skips=2, input_shape=(784,), neuron=128, lr=1e-3, dropout=0.5):
    inputs = tf.keras.Input(shape=input_shape, name='img')
    x = Dense(neuron, activation='relu')(inputs)
    block_output = Dense(neuron, activation='relu')(x)
    
    for i in range(layers):
        layer = block_output
        
        
        for idx in range(skips):
            if idx != skips - 1 or skips == 1:
                layer = Dense(neuron)(layer)
                layer = BatchNormalization()(layer)
                layer = Activation('relu')(layer)
                layer = Dropout(dropout)(layer)
            else:                    
                layer = Dense(neuron)(layer)
                layer = BatchNormalization()(layer)
        block_output = tf.keras.layers.add([layer, block_output])
    
    x = Dense(neuron, activation='relu')(block_output)
    x = Dropout(dropout)(x)
    outputs = Dense(10, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs, name='resnet_10')
    optimizer = Adam(amsgrad=True, lr=lr)
    #optimizer = Adadelta(learning_rate=lr)
    model.compile(optimizer, 'binary_crossentropy', metrics=['accuracy'])
    return model

In [8]:
%%time
# Create the deep ResNet here.
model = create_resnet(layers=10)
model.fit(x_train, y_train, batch_size=128, epochs=20, validation_data=(x_test, y_test))

Train on 60000 samples, validate on 10000 samples
Epoch 1/20
60000/60000 [==============================] - 28s 463us/step - loss: 0.1761 - acc: 0.9467 - val_loss: 0.0432 - val_acc: 0.9855
Epoch 2/20
60000/60000 [==============================] - 18s 305us/step - loss: 0.0518 - acc: 0.9839 - val_loss: 0.0283 - val_acc: 0.9907
Epoch 3/20
60000/60000 [==============================] - 17s 289us/step - loss: 0.0354 - acc: 0.9892 - val_loss: 0.0226 - val_acc: 0.9929
Epoch 4/20
60000/60000 [==============================] - 17s 287us/step - loss: 0.0280 - acc: 0.9915 - val_loss: 0.0212 - val_acc: 0.9934
Epoch 5/20
60000/60000 [==============================] - 17s 286us/step - loss: 0.0227 - acc: 0.9931 - val_loss: 0.0198 - val_acc: 0.9940
Epoch 6/20
60000/60000 [==============================] - 17s 288us/step - loss: 0.0192 - acc: 0.9941 - val_loss: 0.0196 - val_acc: 0.9942
Epoch 7/20
60000/60000 [==============================] - 17s 287us/step - loss: 0.0170 - acc: 0.9947 - val_loss: 0.

In [9]:
plot_model(model, to_file='reports/resnet_10.png')

### Now lets mess with the skip connections. We will make two shallow ResNets that have one and three skip connections.

In [8]:
# Here is a one skip connection block.

x = Dense(128)(block_1_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
block_2_output = tf.keras.layers.add([x, block_1_output])

# Here is a two skip connection block.

x = Dense(128)(block_1_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
block_2_output = tf.keras.layers.add([x, block_1_output])

# Here is a three skip connection block.

x = Dense(128)(block_1_output)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Dropout(0.5)(x)
x = Dense(128)(x)
x = BatchNormalization()(x)
block_2_output = tf.keras.layers.add([x, block_1_output])


In [13]:
%%time
# Now make a full shallow network with the different sized blocks and compare the models. 
# Resnet with skip 1
resnet_skip1 = create_resnet(layers=3, skips=1)
resnet_skip1.fit(x_train, y_train, batch_size=128, epochs=20, validation_data=(x_test, y_test))

Train on 60000 samples, validate on 10000 samples
Epoch 1/20
60000/60000 [==============================] - 9s 153us/step - loss: 0.1146 - acc: 0.9610 - val_loss: 0.0339 - val_acc: 0.9890
Epoch 2/20
60000/60000 [==============================] - 6s 93us/step - loss: 0.0381 - acc: 0.9882 - val_loss: 0.0236 - val_acc: 0.9927
Epoch 3/20
60000/60000 [==============================] - 6s 94us/step - loss: 0.0277 - acc: 0.9915 - val_loss: 0.0209 - val_acc: 0.9937
Epoch 4/20
60000/60000 [==============================] - 6s 95us/step - loss: 0.0216 - acc: 0.9934 - val_loss: 0.0192 - val_acc: 0.9943
Epoch 5/20
60000/60000 [==============================] - 6s 96us/step - loss: 0.0170 - acc: 0.9948 - val_loss: 0.0178 - val_acc: 0.9950
Epoch 6/20
60000/60000 [==============================] - 5s 91us/step - loss: 0.0148 - acc: 0.9954 - val_loss: 0.0164 - val_acc: 0.9951
Epoch 7/20
60000/60000 [==============================] - 5s 87us/step - loss: 0.0132 - acc: 0.9959 - val_loss: 0.0170 - val_ac

In [14]:
plot_model(resnet_skip1, to_file='reports/resnet_skip_1.png')

In [ ]:
%%time
# Resnet with skip 3
resnet_skip3 = create_resnet(layers=3, skips=3)
resnet_skip3.fit(x_train, y_train, batch_size=128, epochs=20, validation_data=(x_test, y_test))

Train on 60000 samples, validate on 10000 samples
Epoch 1/20
60000/60000 [==============================] - 15s 250us/step - loss: 0.1543 - acc: 0.9476 - val_loss: 0.0431 - val_acc: 0.9855
Epoch 2/20
60000/60000 [==============================] - 10s 171us/step - loss: 0.0464 - acc: 0.9852 - val_loss: 0.0297 - val_acc: 0.9906
Epoch 3/20
60000/60000 [==============================] - 10s 171us/step - loss: 0.0322 - acc: 0.9898 - val_loss: 0.0241 - val_acc: 0.9922
Epoch 4/20
60000/60000 [==============================] - 10s 173us/step - loss: 0.0248 - acc: 0.9921 - val_loss: 0.0185 - val_acc: 0.9939
Epoch 5/20
60000/60000 [==============================] - 10s 174us/step - loss: 0.0206 - acc: 0.9934 - val_loss: 0.0165 - val_acc: 0.9945
Epoch 6/20
60000/60000 [==============================] - 10s 175us/step - loss: 0.0173 - acc: 0.9945 - val_loss: 0.0167 - val_acc: 0.9947
Epoch 7/20
60000/60000 [==============================] - 11s 175us/step - loss: 0.0146 - acc: 0.9952 - val_loss: 0.

In [21]:
plot_model(resnet_skip3, to_file='reports/resnet_skip_3.png')

### Now lets create a Functional Model for CIFAR with the same baseline as MNIST.

In [43]:
%%time
# Create the CIFAR baseline here.
#inputs = Input(shape=(input_shape,))
inputs = Input(shape=input_shape_cifar)
output = Dense(64, activation='relu')(inputs)
output = Dense(64, activation='relu')(output)
output = Dense(64, activation='relu')(output)
output = Dense(64, activation='relu')(output)
predictions = Dense(10, activation='softmax')(output)
model = Model(inputs=inputs, outputs=predictions)
model.compile(optimizer=SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train_cifar, y_train_cifar, epochs=20, batch_size=64, validation_data=(x_test_cifar, y_test_cifar))

Train on 50000 samples, validate on 10000 samples
Epoch 1/20
50000/50000 [==============================] - 13s 252us/step - loss: 1.8652 - acc: 0.3213 - val_loss: 1.7420 - val_acc: 0.3654
Epoch 2/20
50000/50000 [==============================] - 6s 123us/step - loss: 1.6760 - acc: 0.3970 - val_loss: 1.7167 - val_acc: 0.3837
Epoch 3/20
50000/50000 [==============================] - 6s 125us/step - loss: 1.6008 - acc: 0.4249 - val_loss: 1.6314 - val_acc: 0.4105
Epoch 4/20
50000/50000 [==============================] - 6s 121us/step - loss: 1.5543 - acc: 0.4436 - val_loss: 1.5396 - val_acc: 0.4518
Epoch 5/20
50000/50000 [==============================] - 6s 123us/step - loss: 1.5166 - acc: 0.4563 - val_loss: 1.5864 - val_acc: 0.4318
Epoch 6/20
50000/50000 [==============================] - 6s 121us/step - loss: 1.4908 - acc: 0.4670 - val_loss: 1.5516 - val_acc: 0.4447
Epoch 7/20
50000/50000 [==============================] - 6s 120us/step - loss: 1.4634 - acc: 0.4760 - val_loss: 1.5016 -

### Now lets create a ResNet for CIFAR with the same layers and compare it with MNIST.
Why does our accuracy change so much? 

In [93]:
%%time
# Create the CIFAR ResNet here.
resnet_cifar = create_resnet(input_shape=input_shape_cifar)
resnet_cifar.fit(x_train_cifar, y_train_cifar, epochs=20, batch_size=64, validation_data=(x_test_cifar, y_test_cifar))

Train on 50000 samples, validate on 10000 samples
Epoch 1/20
50000/50000 [==============================] - 54s 1ms/step - loss: 0.3102 - acc: 0.8985 - val_loss: 0.2756 - val_acc: 0.9007
Epoch 2/20
50000/50000 [==============================] - 17s 335us/step - loss: 0.2836 - acc: 0.9008 - val_loss: 0.2713 - val_acc: 0.9025
Epoch 3/20
50000/50000 [==============================] - 17s 334us/step - loss: 0.2731 - acc: 0.9018 - val_loss: 0.2608 - val_acc: 0.9038
Epoch 4/20
50000/50000 [==============================] - 17s 337us/step - loss: 0.2654 - acc: 0.9034 - val_loss: 0.2550 - val_acc: 0.9052
Epoch 5/20
50000/50000 [==============================] - 17s 331us/step - loss: 0.2590 - acc: 0.9049 - val_loss: 0.2533 - val_acc: 0.9064
Epoch 6/20
50000/50000 [==============================] - 17s 338us/step - loss: 0.2534 - acc: 0.9064 - val_loss: 0.2440 - val_acc: 0.9074
Epoch 7/20
50000/50000 [==============================] - 17s 339us/step - loss: 0.2490 - acc: 0.9073 - val_loss: 0.24

### Now lets create a custom ResNet.
Change the layer widths, dropout layers, batch sizes, and skip connections to see what we could do to make the ResNet better. Use the knowledge you gained from Lab 3 to do this.

In [1]:
# Create your own ResNet here.

def get_dense(neuron):
    result = Dense(neuron)
    return result

#def get_conv():
    

def customize_resnet(layers=3, skips=2, lr=1e-3, dropout=0.5, layer_base=get_dense, **kwargs):
    input_shape = input_shape_cifar
    inputs = tf.keras.Input(shape=input_shape, name='img')
    #x = Dense(128)(inputs)
    x = layer_base(128)(inputs)
    x = Activation('relu')(x)
    #block_output = layer_base(x)
    block_output = layer_base(128)(x)
    block_output = Activation('relu')(x)
    
    for i in range(layers):
        layer = block_output
        
        
        for idx in range(skips):
            if idx != skips - 1 or skips == 1:
                #layer = layer_base(layer)
                layer = get_dense(128)(layer)
                layer = BatchNormalization()(layer)
                layer = Activation('relu')(layer)
                layer = Dropout(dropout)(layer)
            else:                    
                #layer = layer_base(layer)
                layer = get_dense(128)(layer)
                layer = BatchNormalization()(layer)
        block_output = tf.keras.layers.add([layer, block_output])
    
    #x = layer_base(block_output)
    x = get_dense(128)(block_output)
    x = Activation('relu')(x)
    x = Dropout(dropout)(x)
    outputs = Dense(10, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs, name='resnet_10')
    optimizer = Adam(amsgrad=True, lr=lr)
    #optimizer = Adadelta(learning_rate=lr)
    model.compile(optimizer, 'binary_crossentropy', metrics=['accuracy'])
    return model

In [2]:
conv = Conv2D(16, kernel_size=1, strides=2,
                  padding='same',
                  kernel_initializer='he_normal',
                  #kernel_regularizer=l2(1e-4)
             )

custom_model = customize_resnet(layers=3, skips=2, lr=1e-3, dropout=0.4, neuron=128)
best_save = ModelCheckpoint('models/best_cifar.hdf5', save_best_only=True, monitor='val_acc', mode='max')
history = custom_model.fit(x_train_cifar, y_train_cifar, epochs=20, batch_size=64, validation_data=(x_test_cifar, y_test_cifar), callbacks=[best_save])
pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True)
plt.show()

NameError: name 'Conv2D' is not defined

In [49]:
custom_model.load_weights("models/best_cifar.hdf5")
y_pred = custom_model.predict(x_test_cifar)
y_classes = y_pred.argmax(axis=-1).reshape(-1, 1)
#aa = tf.keras.utils.to_categorical(y_classes, 10)
#print(y_pred[1])
print(y_classes[0:9])
print(y_test_cifar_origin[0:9])
print("Accuracy:", accuracy_score(aa, y_test_cifar))

[[3]
 [9]
 [0]
 [0]
 [4]
 [6]
 [1]
 [6]
 [2]]
[[3]
 [8]
 [8]
 [0]
 [6]
 [6]
 [1]
 [6]
 [3]]
Accuracy: 0.5053


In [41]:
y_test_cifar

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 1., 0., 0.]], dtype=float32)